# Thumbnail Visualization Pipeline — Health Category

유튜브 **헬스/운동 카테고리** 채널의 썸네일 이미지를 시각화하는 파이프라인입니다.  
채널당 영상 1개씩 수집한 데이터를 기반으로, 비전 모델 임베딩의 전체 분포를 탐색합니다.

## 데이터 개요

| 파일 | 설명 |
|---|---|
| `YT_dataset_health.csv` | 헬스 채널 영상 메타데이터 + 썸네일 로컬 경로 |

## 파이프라인 단계

```
Step 1: prepare   → 썸네일 경로 검증, dataset_ready.csv 생성
Step 2: extract   → 비전 모델로 썸네일 임베딩 추출 (DINOv2 / SigLIP2)
Step 3: visualize → UMAP / t-SNE 2D 시각화
```

---
## 라이브러리 & 전역 설정

파이프라인 전체에서 사용하는 패키지를 임포트하고 경로·하이퍼파라미터 상수를 정의합니다.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import re
import sys
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

# ── 작업 디렉터리를 프로젝트 루트로 고정 (thumbnail_path 상대 경로 기준) ──
if not (Path.cwd() / "YT_dataset_health.csv").exists():
    _candidates = list(Path.cwd().rglob("YT_dataset_health.csv"))
    if _candidates:
        os.chdir(_candidates[0].parent)
print(f"CWD: {Path.cwd()}")

# ── 경로 설정 ──────────────────────────────────────────────────────────────
YOUTUBE_CSV    = Path("YT_dataset_health.csv")   # 02.data_collection 결과
OUTPUT_DIR     = Path("artifacts")
EMBEDDINGS_DIR = OUTPUT_DIR / "embeddings"
VIZ_DIR        = OUTPUT_DIR / "visualizations"

# ── 모델 설정 ─────────────────────────────────────────────────────────────
MODEL_ALIASES = {
    "dinov2-base":  "facebook/dinov2-base",
    "siglip2-base": "google/siglip2-base-patch16-224",
}
MODELS_TO_RUN = "dinov2-base,siglip2-base"   # 쉼표로 여러 모델 지정 가능
BATCH_SIZE    = 32
DEVICE        = "auto"   # "auto" | "cpu" | "cuda"

# ── 차원 축소 파라미터 ─────────────────────────────────────────────────────
UMAP_NEIGHBORS    = 20
UMAP_MIN_DIST     = 0.1
TSNE_PERPLEXITIES = [5, 15, 30]

print("설정 완료")
print(f"  youtube_csv  : {YOUTUBE_CSV}")
print(f"  output_dir   : {OUTPUT_DIR}")

---
## 공통 유틸리티 함수

파이프라인 전 단계에서 재사용되는 파일 I/O, 경로 처리 함수입니다.

In [ ]:
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def read_csv_rows(path: Path, encoding: str = "utf-8-sig") -> List[Dict[str, str]]:
    with path.open("r", encoding=encoding, newline="") as f:
        return list(csv.DictReader(f))


def write_csv_rows(
    path: Path,
    fieldnames: Sequence[str],
    rows: Iterable[Dict],
) -> None:
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


print("유틸리티 함수 정의 완료")

---
## Step 1: 데이터 준비 (`prepare`)

### 목적
`YT_dataset_health.csv`를 읽고 **썸네일 파일이 실제로 존재하는 행만** 추려  
`artifacts/dataset_ready.csv`를 생성합니다.

> Shorts 필터링은 `02.data_collection` 단계에서 이미 완료되었으므로 여기서는 생략합니다.

### 출력 파일

| 파일 | 용도 |
|---|---|
| `dataset_ready.csv` | 썸네일 확인 완료, 임베딩 추출 대상 |
| `prepare_summary.json` | 처리 통계 |

In [ ]:
def prepare_dataset(
    youtube_csv: Path = YOUTUBE_CSV,
    output_dir: Path = OUTPUT_DIR,
) -> None:
    ensure_dir(output_dir)
    rows = read_csv_rows(youtube_csv)

    ready, skipped_no_path, skipped_missing = [], 0, 0
    for row in rows:
        thumb_str = (row.get("thumbnail_path") or "").strip()
        if not thumb_str:
            skipped_no_path += 1
            continue
        thumb = Path(thumb_str)
        if not thumb.exists():
            skipped_missing += 1
            continue
        ready.append({
            "channel_name":   (row.get("channel_name")  or "").strip(),
            "channel_id":     (row.get("channel_id")    or "").strip(),
            "video_id":       (row.get("video_id")      or "").strip(),
            "title":          (row.get("title")         or "").strip(),
            "published_at":   (row.get("published_at")  or "").strip(),
            "duration":       (row.get("duration")      or "").strip(),
            "thumbnail_path": thumb_str,
            "thumbnail_url":  (row.get("thumbnail_url") or "").strip(),
        })

    fieldnames = [
        "channel_name", "channel_id", "video_id",
        "title", "published_at", "duration",
        "thumbnail_path", "thumbnail_url",
    ]
    write_csv_rows(output_dir / "dataset_ready.csv", fieldnames, ready)

    summary = {
        "total_input":       len(rows),
        "skipped_no_path":   skipped_no_path,
        "skipped_missing":   skipped_missing,
        "ready_count":       len(ready),
    }
    with (output_dir / "prepare_summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"입력: {len(rows)}개  /  경로 없음: {skipped_no_path}  /  파일 없음: {skipped_missing}")
    print(f"임베딩 대상: {len(ready)}개")
    print(f"저장: {output_dir / 'dataset_ready.csv'}")


print("prepare 함수 정의 완료")

In [ ]:
# ── Step 1 실행 ────────────────────────────────────────────────────────────
prepare_dataset(
    youtube_csv=YOUTUBE_CSV,
    output_dir=OUTPUT_DIR,
)

---
## Step 2: 임베딩 추출 (`extract`)

### 목적
HuggingFace 비전 모델로 썸네일 이미지를 **고차원 벡터(임베딩)** 로 변환합니다.

| 모델 | 임베딩 차원 | 특징 |
|---|---|---|
| `dinov2-base` | 768 | 자기지도학습(DINO v2), 이미지 구조 포착 강점 |
| `siglip2-base` | 768 | 이미지-텍스트 대조학습, 의미적 특징 포착 강점 |

### 출력 파일 (모델별)

| 파일 | 내용 |
|---|---|
| `embeddings.npy` | (N, 768) 임베딩 행렬 |
| `metadata.csv` | 영상별 메타데이터 |
| `run_info.json` | 실행 설정 정보 |

> GPU가 있다면 `DEVICE = "cuda"` 로 변경하면 훨씬 빠릅니다.

In [ ]:
import numpy as np
from PIL import Image


@dataclass
class LoadedModel:
    model_id:  str
    alias:     str
    processor: object
    model:     object
    device:    str


def _resolve_device(requested: str) -> str:
    if requested != "auto":
        return requested
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
    except Exception:
        pass
    return "cpu"


def _resolve_model_id(alias_or_id: str) -> Tuple[str, str]:
    alias_or_id = alias_or_id.strip()
    if alias_or_id in MODEL_ALIASES:
        return alias_or_id, MODEL_ALIASES[alias_or_id]
    safe_alias = re.sub(r"[^a-zA-Z0-9._-]+", "_", alias_or_id)
    return safe_alias, alias_or_id


def _load_transformers_model(alias_or_id: str, device: str) -> LoadedModel:
    from transformers import AutoImageProcessor, AutoModel, AutoProcessor
    alias, model_id = _resolve_model_id(alias_or_id)
    model = AutoModel.from_pretrained(model_id)
    try:
        processor = AutoProcessor.from_pretrained(model_id)
    except Exception:
        processor = AutoImageProcessor.from_pretrained(model_id)
    model = model.to(device).eval()
    return LoadedModel(model_id=model_id, alias=alias,
                       processor=processor, model=model, device=device)


def _extract_batch_embeddings(loaded: LoadedModel, images: list) -> list:
    import torch
    inputs = loaded.processor(images=images, return_tensors="pt")
    inputs = {k: v.to(loaded.device) for k, v in inputs.items()
              if isinstance(v, torch.Tensor)}
    with torch.no_grad():
        outputs = loaded.model(**inputs)
    if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
        feats = outputs.pooler_output
    else:
        feats = outputs.last_hidden_state[:, 0, :]
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().float().numpy().tolist()


def extract_embeddings(
    dataset_csv: Path,
    output_root: Path = EMBEDDINGS_DIR,
    models: str = MODELS_TO_RUN,
    batch_size: int = BATCH_SIZE,
    device: str = DEVICE,
) -> None:
    ensure_dir(output_root)
    rows = read_csv_rows(dataset_csv, encoding="utf-8")

    if not rows:
        raise RuntimeError(f"데이터가 없습니다: {dataset_csv}")

    device_str  = _resolve_device(device)
    model_specs = [m.strip() for m in models.split(",") if m.strip()]
    print(f"추출 대상: {len(rows)}행 | 디바이스: {device_str} | 모델: {model_specs}")

    for model_name in model_specs:
        print(f"\n[{model_name}] 모델 로드 중...")
        loaded    = _load_transformers_model(model_name, device=device_str)
        model_dir = output_root / loaded.alias
        ensure_dir(model_dir)

        all_embs, all_meta = [], []
        batch_imgs, batch_meta_buf = [], []

        def flush():
            if not batch_imgs:
                return
            for i, emb in enumerate(_extract_batch_embeddings(loaded, batch_imgs)):
                all_embs.append(emb)
                all_meta.append(batch_meta_buf[i])
            batch_imgs.clear()
            batch_meta_buf.clear()

        missing_thumb = 0
        for row in rows:
            thumb = Path(row["thumbnail_path"])
            if not thumb.exists():
                missing_thumb += 1
                continue
            try:
                img = Image.open(thumb).convert("RGB")
            except Exception:
                continue
            batch_imgs.append(img)
            batch_meta_buf.append({
                "video_id":       row.get("video_id", ""),
                "channel_id":     row.get("channel_id", ""),
                "channel_name":   row.get("channel_name", ""),
                "published_at":   row.get("published_at", ""),
                "duration":       row.get("duration", ""),
                "thumbnail_path": row.get("thumbnail_path", ""),
            })
            if len(batch_imgs) >= batch_size:
                flush()
        flush()

        if missing_thumb:
            print(f"  썸네일 파일 없음(건너뜀): {missing_thumb}개")
        if not all_embs:
            print(f"  [{model_name}] 임베딩 없음, 건너뜀")
            continue

        embeddings = np.stack(all_embs, axis=0)
        np.save(model_dir / "embeddings.npy", embeddings)

        meta_fields = ["video_id", "channel_id", "channel_name",
                       "published_at", "duration", "thumbnail_path"]
        write_csv_rows(model_dir / "metadata.csv", meta_fields, all_meta)

        run_info = {
            "alias": loaded.alias, "model_id": loaded.model_id, "device": device_str,
            "batch_size": batch_size, "rows_input": len(rows),
            "rows_embedded": int(embeddings.shape[0]),
            "embedding_dim": int(embeddings.shape[1]),
        }
        with (model_dir / "run_info.json").open("w", encoding="utf-8") as f:
            json.dump(run_info, f, ensure_ascii=False, indent=2)

        print(f"  임베딩 shape: {embeddings.shape} → {model_dir / 'embeddings.npy'}")


print("extract 함수 정의 완료")

In [ ]:
# ── Step 2 실행 ────────────────────────────────────────────────────────────
# GPU 사용 시: DEVICE = "cuda"
# 특정 모델만: MODELS_TO_RUN = "dinov2-base"

extract_embeddings(
    dataset_csv=OUTPUT_DIR / "dataset_ready.csv",
    output_root=EMBEDDINGS_DIR,
    models=MODELS_TO_RUN,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

---
## Step 3: 시각화 (`visualize`)

### 목적
각 모델의 임베딩을 UMAP / t-SNE로 2차원 축소하여  
**헬스 카테고리 썸네일의 전체 분포 형태**를 탐색합니다.

### 출력 파일 (모델별)

| 파일 | 내용 |
|---|---|
| `umap.png` | UMAP 2D scatter |
| `tsne_perp{N}.png` | t-SNE scatter (perplexity N별) |
| `visualize_summary.json` | 실행 요약 |

In [ ]:
def _scatter(path: Path, coords, title: str, color: str = "#4C72B0") -> None:
    """단색 2D scatter plot 저장."""    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(coords[:, 0], coords[:, 1], s=14, alpha=0.55, color=color, linewidths=0)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("dim-1")
    ax.set_ylabel("dim-2")
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)
    print(f"  저장: {path}")


def visualize_model(
    model_alias: Optional[str] = None,
    embeddings_root: Path = EMBEDDINGS_DIR,
    output_dir: Path = VIZ_DIR,
    umap_neighbors: int = UMAP_NEIGHBORS,
    umap_min_dist: float = UMAP_MIN_DIST,
    tsne_perplexities: List[int] = TSNE_PERPLEXITIES,
) -> None:
    """model_alias 미지정 시 EMBEDDINGS_DIR 내 첫 번째 모델 자동 선택."""    import numpy as np
    import umap as umap_lib
    from sklearn.manifold import TSNE

    model_dirs = sorted([p for p in embeddings_root.iterdir() if p.is_dir()])
    if not model_dirs:
        raise RuntimeError(f"임베딩 디렉터리가 없습니다: {embeddings_root}")

    if model_alias:
        target_dirs = [d for d in model_dirs if d.name == model_alias]
        if not target_dirs:
            raise ValueError(f"'{model_alias}' 를 찾을 수 없습니다. 가능한 모델: {[d.name for d in model_dirs]}")
    else:
        target_dirs = model_dirs   # 전체 모델 시각화

    ensure_dir(output_dir)

    for model_dir in target_dirs:
        emb_path  = model_dir / "embeddings.npy"
        meta_path = model_dir / "metadata.csv"
        if not emb_path.exists():
            print(f"[{model_dir.name}] embeddings.npy 없음, 건너뜀")
            continue

        X = np.load(emb_path)
        print(f"\n[{model_dir.name}] 임베딩 shape: {X.shape}")

        model_out = output_dir / model_dir.name
        ensure_dir(model_out)

        # ── UMAP ─────────────────────────────────────────────────────────
        print(f"[{model_dir.name}] UMAP 축소 중...")
        reducer = umap_lib.UMAP(
            n_components=2, n_neighbors=umap_neighbors,
            min_dist=umap_min_dist, metric="cosine", random_state=42,
        )
        umap_coords = reducer.fit_transform(X)
        _scatter(model_out / "umap.png", umap_coords,
                 title=f"UMAP — {model_dir.name} (n={X.shape[0]})")

        # ── t-SNE ─────────────────────────────────────────────────────────
        for p in tsne_perplexities:
            if p >= X.shape[0]:
                print(f"  t-SNE perplexity={p} 건너뜀 (샘플 수 부족)")
                continue
            print(f"[{model_dir.name}] t-SNE (perplexity={p}) 축소 중...")
            coords = TSNE(n_components=2, perplexity=p, init="pca",
                          learning_rate="auto", random_state=42).fit_transform(X)
            _scatter(model_out / f"tsne_perp{p}.png", coords,
                     title=f"t-SNE perp={p} — {model_dir.name} (n={X.shape[0]})")

        summary = {
            "model": model_dir.name,
            "n_samples": int(X.shape[0]),
            "embedding_dim": int(X.shape[1]),
            "umap_params": {"n_neighbors": umap_neighbors, "min_dist": umap_min_dist},
            "tsne_perplexities": tsne_perplexities,
        }
        with (model_out / "visualize_summary.json").open("w", encoding="utf-8") as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"\n시각화 완료 → {output_dir}")


print("visualize 함수 정의 완료")

In [ ]:
# ── Step 3 실행 ────────────────────────────────────────────────────────────
# model_alias=None → EMBEDDINGS_DIR 내 모든 모델 한번에 시각화
# 특정 모델만: model_alias="dinov2-base"

visualize_model(
    model_alias=None,
    embeddings_root=EMBEDDINGS_DIR,
    output_dir=VIZ_DIR,
    umap_neighbors=UMAP_NEIGHBORS,
    umap_min_dist=UMAP_MIN_DIST,
    tsne_perplexities=TSNE_PERPLEXITIES,
)

---
## 결과 인라인 확인

생성된 UMAP / t-SNE 이미지를 노트북 안에서 바로 확인합니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


def show_images(paths, cols: int = 2, title_prefix: str = "") -> None:
    paths = [Path(p) for p in paths if Path(p).exists()]
    if not paths:
        print("표시할 이미지가 없습니다.")
        return
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 7, rows * 5.5))
    axes = [axes] if rows * cols == 1 else list(
        axes.flat if hasattr(axes, "flat") else [axes]
    )
    for ax, p in zip(axes, paths):
        ax.imshow(mpimg.imread(p))
        ax.set_title(f"{title_prefix}{p.stem}", fontsize=10)
        ax.axis("off")
    for ax in axes[len(paths):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# ── 모든 모델의 UMAP / t-SNE 이미지 표시 ──────────────────────────────────
for model_dir in sorted(VIZ_DIR.iterdir()):
    if not model_dir.is_dir():
        continue
    imgs = sorted(model_dir.glob("*.png"))
    if imgs:
        print(f"\n=== {model_dir.name} ===")
        show_images(imgs, cols=2)